# Component graph construction and manipulation

This complete native tutorial follows all 42 cells of the locked source tutorial. Compiler owns typed graph construction, mapping, copying and expansion. Core owns validation. Drawer presents the graph. The final example uses `circuit.append(op)`.

Adapted from Google LLC, Apache-2.0, Qualtran v0.7.0 `qualtran/_infra/composite_bloq.ipynb`, commit `bcb17bb25360ed68ce3bc084645db3e929799389`, blob `1fe7ddcab53ddd9a909417c8994d643d8f9d8ff4`.


In [ ]:
from dataclasses import replace
import numpy as np
from examples.algorithms.component_construction_tutorial import (
    BoundComponentExample,
    compose_parallel,
    labelled_connections,
    two_cnot_program,
    wrap_program,
)
from examples.algorithms.register_graphs import parallel_program
from flagquantum.compiler.registers.component_builder import ComponentProgramBuilder
from flagquantum.compiler.registers.component_binding_map import map_register_bindings
from flagquantum.compiler.registers.component_graph import (
    copy_component_program,
    iter_component_calls,
)
from flagquantum.compiler.registers.component_expansion import (
    expand_component_calls,
    expand_component_calls_once,
)
from flagquantum.core.ir.program import verify_program_record
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.component_score import component_register_score
from graphviz import Source
from flagquantum.drawer.register_graph import export_register_graph_dot


def show_program(record):
    return Source(export_register_graph_dot(component_register_score(record)))


## Build a component graph
Two CNOT calls can connect the same operands or reverse their roles. The builder consumes each handle and returns fresh outputs in signature order.


In [ ]:
straight = two_cnot_program(crossed=False)
show_program(straight)


In [ ]:
crossed = two_cnot_program(crossed=True)
show_program(crossed)


The second crossed CNOT returns its control first. Binding that output to `q0` makes the output order differ from physical wire order. The graph preserves this distinction.


## Explicit decomposition
A native operation supplies its owned Program decomposition. Declarations, graph definitions and physical execution remain distinct.


In [ ]:
def decompose_two_cnot():
    return two_cnot_program(crossed=True)


In [ ]:
two_cnot = decompose_two_cnot()
print(two_cnot.functions[0].registers)


In [ ]:
show_program(two_cnot)


## Debug text
Inspect every immediate call with its predecessor and successor register handles.


In [ ]:
for position, inputs, outputs in iter_component_calls(crossed):
    print(position, inputs, outputs)


## Validate the decomposition
Core verification checks the whole typed Program, including calls, register compatibility, ownership and return bindings.


In [ ]:
assert verify_program_record(crossed) == ()
assert verify_program_record(straight) == ()


Use the same Core verifier on the complete explicit decomposition.


In [ ]:
assert verify_program_record(decompose_two_cnot()) == ()
print("Checks out!")


## Iterate connections
Inspect directed predecessor-to-successor connections independently of rendering.


In [ ]:
for predecessor, successor in labelled_connections(crossed):
    print(predecessor, "->", successor)


## Copy
Copying rebuilds the call graph and preserves connectivity. Shared immutable definitions remain shared.


In [ ]:
parallel = parallel_program()
copied = copy_component_program(parallel)
assert copied == parallel and copied is not parallel
show_program(copied)


Use explicit presentation labels to show copied occurrence identities. The source demonstrates an offset by monkeypatching builder construction. Here that offset belongs to presentation.


In [ ]:
def copied_labels(record, offset):
    return labelled_connections(record, offset=offset)


Compare original labels with a copy labelled from 100. Connectivity and register coordinates remain equal.


In [ ]:
original_connections = labelled_connections(parallel)
copy_connections = copied_labels(copied, 100)
for original, copied_connection in zip(
    original_connections, copy_connections, strict=True
):
    print(original, copied_connection)


## Iterate calls and map handles
Rebuild the parallel example manually using the original call order and the shared scalar/array mapping operation. Mapping does not consume handles. Inserting each call consumes inputs through the ordinary builder.


In [ ]:
root = parallel.functions[0]
bb = ComponentProgramBuilder(Signature(root.registers), parallel.functions[1:])
first_inputs = next(iter_component_calls(parallel))[1]
replacements = [(first_inputs["reg"], bb.inputs["reg"])]
for position, inputs, old_outputs in iter_component_calls(parallel):
    callee = root.blocks[0].operations[position].callee
    new_outputs = bb.call_tuple(callee, map_register_bindings(inputs, replacements))
    replacements.extend(zip(old_outputs, new_outputs, strict=True))
manual_copy = bb.finalize(map_register_bindings({"reg": old_outputs[0]}, replacements))
assert manual_copy == parallel


## Inline during construction
A regular call keeps the child definition nested. `inline_calls` inserts its immediate contents into the live builder and retains deeper children.


In [ ]:
nested_pair = compose_parallel()
show_program(nested_pair)


In [ ]:
inline_second = compose_parallel(inline_at=frozenset({1}))
show_program(inline_second)


In [ ]:
inline_first = compose_parallel(inline_at=frozenset({0}))
show_program(inline_first)


In [ ]:
inline_middle = compose_parallel(count=3, inline_at=frozenset({1}))
show_program(inline_middle)


## Expand an existing graph
Expansion removes one selected level after construction. A predicate chooses occurrences by retained labels.


In [ ]:
three_children = compose_parallel(count=3)
three_wrapped = wrap_program(three_children, name="three_parallel")
show_program(three_wrapped)


In [ ]:
flat_three, flat_labels = expand_component_calls_once(three_wrapped)
show_program(flat_three)


Decomposing child definitions leaves three parent calls. It does not splice those child calls into the parent.


In [ ]:
# A decomposed child remains one nested call until it is expanded.
root = flat_three.functions[0]
callee = root.blocks[0].operations[0].callee
child = next(function for function in flat_three.functions if function.name == callee)
nested_decompositions = replace(
    flat_three,
    functions=(
        replace(
            root,
            blocks=(
                replace(
                    root.blocks[0],
                    operations=tuple(
                        replace(op, callee="parallel_decomposed")
                        for op in root.blocks[0].operations
                    ),
                ),
            ),
        ),
        *flat_three.functions[1:],
        replace(child, name="parallel_decomposed"),
    ),
)
assert verify_program_record(nested_decompositions) == ()
show_program(nested_decompositions)


Expand the immediate children to splice their split, X and join operations into the parent.


In [ ]:
expanded_children, expanded_labels = expand_component_calls_once(
    flat_three, instance_ids=flat_labels
)
show_program(expanded_children)


The source predicate selects retained occurrence label 1. After removing the wrapper, that is the first child. Explicit labels preserve the same selection across passes.


In [ ]:
# Use the original retained instance label, not a position in the new list.
selective, selective_labels = expand_component_calls_once(
    flat_three,
    instance_ids=flat_labels,
    predicate=lambda label, operation: label == 1,
)
show_program(selective)


Repeated expansion stops when no typed child can expand. Actual X primitives and the source synthetic T model are reported separately.


In [ ]:
fully_flat, final_labels = expand_component_calls(three_wrapped)
show_program(fully_flat)
import flagquantum as fq

circuit = fq.Circuit(3)
circuit.append(BoundComponentExample(fully_flat, ((0, 1, 2),)))
primitive_counts = {"x": len(circuit.to_ir().instructions)}
# The original TestAtom tensor is X, but its synthetic model costs 100 T.
source_model_estimate = {"t": 9 * 100}
print("Measured primitives", primitive_counts)
print("Source synthetic model", source_model_estimate)
